# Knee v6: ConvNeXt 5-fold retrain, second seed (CONFIG convnext2)
Same recipe as the first ConvNeXt run (`convnext_tiny.in12k_ft_in1k`, `--head tokens`, 160 px cache, 12 epochs, folds 0-2 on GPU 0 and 3-4 on GPU 1, ~5 h wall) with a different `--seed`.
Folds come from GroupKFold on site, so they do not depend on the seed: fold k validates on the same studies as in the first run, and the two OOF files can be ensembled directly.
Outputs: `models_v6convnext2/fold{i}_v6convnext2.pt`, `oof_v6convnext2.csv`.
Inputs: competition data, `knee-mri-pipeline` (knee_pipeline_v6, with the patched model.py), caching output, training output (oof_all.csv),
and the FIRST ConvNeXt run's output (oof_v6convnext.csv) for the seed-ensemble OOF check. GPU T4 x2, internet ON, Save & Run All.

In [1]:
import os, sys, json, subprocess, time, threading, shutil
from pathlib import Path
import numpy as np, pandas as pd

CONFIG = "convnext2"                                    # tag v6convnext2: never collides with the first run's v6convnext files
SEED = int(os.environ.get("KNEE_SEED", "43"))           # first run used the default 42
BACKBONE = "convnext_tiny.in12k_ft_in1k"
TAG = f"v6{CONFIG}"
W = Path(os.environ.get("KNEE_WORK", "/kaggle/working"))
CODE   = Path(os.environ.get("KNEE_CODE",   "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"))
DATA   = Path(os.environ.get("KNEE_DATA",   "/kaggle/input/competitions/rsna-knee-abnormality-detection"))
CACHE  = Path(os.environ.get("KNEE_CACHE",  "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache"))
LABELS = Path(os.environ.get("KNEE_LABELS", "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv"))
BASE   = Path(os.environ.get("KNEE_BASE",   "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv"))
for name, p in dict(CODE=CODE, DATA=DATA, CACHE=CACHE, LABELS=LABELS, BASE=BASE).items():
    assert p.exists(), f"{name} not found: {p}"
for f in ("geom.py", "tests/smoke_v6.py"):
    assert (CODE / f).exists(), f"{f} missing: dataset does not hold knee_pipeline_v6"
print("CONFIG", CONFIG, "| TAG", TAG, "| seed", SEED, "| backbone", BACKBONE)

CONFIG convnext2 | TAG v6convnext2 | seed 43 | backbone convnext_tiny.in12k_ft_in1k


In [2]:
for t in ("tests/test_geom.py", "tests/smoke_v6.py"):
    r = subprocess.run([sys.executable, str(CODE / t)], capture_output=True, text=True, cwd=str(CODE))
    print(f"== {t} (exit {r.returncode})\n{r.stdout[-800:]}")
    assert r.returncode == 0, f"{t} failed:\n{r.stderr[-1500:]}"
EXTRA = ["--head", "tokens", "--backbone", BACKBONE, "--seed", str(SEED)]
print("extra train args:", EXTRA)

== tests/test_geom.py (exit 0)
canonical orientation identical for 24 acquisition conventions
right/left mirror normalisation OK for 3 planes
laterality rules OK
ALL GEOM TESTS PASS

== tests/smoke_v6.py (exit 0)
head=series: forward/backward OK, absent-series masking OK, params 5.69M
head=tokens: forward/backward OK, absent-series masking OK, params 5.18M
backbone convnext_tiny.in12k_ft_in1k: OK | vit_size None | params 28.9M
backbone efficientnet_b2: OK | vit_size None | params 8.9M
backbone vit_small_patch14_dinov2.lvd142m: OK | vit_size 168 | params 22.5M
backbone vit_small_patch16_224.dino: OK | vit_size 160 | params 22.6M
backbone deit_tiny_patch16_224.fb_in1k: OK | vit_size 160 | params 6.4M
v5-style state_dict loads into head='series'
KneeDS geometry parity OK
ALL V6 SMOKE TESTS PASS

extra train args: ['--head', 'tokens', '--backbone', 'convnext_tiny.in12k_ft_in1k', '--seed', '43']


In [3]:
BASE_ARGS = ["--cache_dir", str(CACHE), "--labels", str(LABELS), "--epochs", "12", "--pretrained",
             "--workers", "2", "--train_slices", "16", "--bs", "4", "--accum", "2"]
JOBS = {0: (f"{TAG}_g0", ["0", "1", "2"]), 1: (f"{TAG}_g1", ["3", "4"])}
LOGS = {}
def run(gpu, name, folds):
    out = W / name
    if (out / "oof.csv").exists():
        print(f"[{name}] oof.csv exists, skipped"); return
    cmd = [sys.executable, str(CODE / "train.py")] + BASE_ARGS + EXTRA + ["--only_folds"] + folds + ["--out", str(out)]
    LOGS[name] = W / f"{name}.log"
    print(f"[{name}] GPU {gpu}: {' '.join(cmd)}", flush=True)
    with open(LOGS[name], "w") as fh:
        rc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)), cwd=str(CODE)).wait()
    print(f"[{name}] exit code {rc}", flush=True)
threads = [threading.Thread(target=run, args=(g, n, f)) for g, (n, f) in JOBS.items()]
for th in threads: th.start()
t0 = time.time()
while any(th.is_alive() for th in threads):
    time.sleep(900)
    last = {}
    for n, f in LOGS.items():
        ls = [l for l in f.read_text().splitlines() if "ep " in l] if f.exists() else []
        last[n] = ls[-1] if ls else "(none yet)"
    print(f"[{(time.time()-t0)/3600:.2f} h] {json.dumps(last, indent=1)}", flush=True)
for th in threads: th.join()
print("all jobs finished")

[v6convnext2_g0] GPU 0: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 4 --accum 2 --head tokens --backbone convnext_tiny.in12k_ft_in1k --seed 43 --only_folds 0 1 2 --out /kaggle/working/v6convnext2_g0
[v6convnext2_g1] GPU 1: /usr/bin/python3 /kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline/train.py --cache_dir /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache --labels /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv --epochs 12 --pretrained --workers 2 --train_slices 16 --bs 4 --accum 2 --head tokens --backbone convnext_tiny.in12k_ft_in1k --seed 43 --only_folds 3 4 --out /kaggle/working/v6convnext2_g1
[0.25 h] {
 "v6convnext2_g0": "[f0] ep 1/12 loss 0.8035 | va

In [4]:
# ---- collect: unique model names + merged OOF ----
M = W / f"models_{TAG}"; M.mkdir(exist_ok=True)
for name, folds in JOBS.values():
    for k in folds:
        src = W / name / f"fold{k}.pt"
        assert src.exists(), f"missing {src}: a job failed, do not use this output"
        shutil.copy(src, M / f"fold{k}_{TAG}.pt")
print(sorted(p.name for p in M.glob("*.pt")))
assert len(list(M.glob("*.pt"))) == 5
LAB = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
o0, o1 = pd.read_csv(W / JOBS[0][0] / "oof.csv"), pd.read_csv(W / JOBS[1][0] / "oof.csv")
oof = o0.copy()
for l in LAB:
    oof[f"p_{l}"] = o0[f"p_{l}"].fillna(o1[f"p_{l}"])
assert oof[[f"p_{l}" for l in LAB]].notna().all().all(), "some studies have no OOF prediction"
assert (o0.fold == o1.fold).all(), "fold assignment differs between the two jobs"
oof.to_csv(W / f"oof_{TAG}.csv", index=False)
print("saved", f"oof_{TAG}.csv", oof.shape)

['fold0_v6convnext2.pt', 'fold1_v6convnext2.pt', 'fold2_v6convnext2.pt', 'fold3_v6convnext2.pt', 'fold4_v6convnext2.pt']
saved oof_v6convnext2.csv (4407, 40)


In [5]:
# ---- pooled 5-fold analysis on the FIXED old labels (confident: y>=0.9 or <=0.1) ----
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
KEY4 = LAB[:4]
base = pd.read_csv(BASE)
assert len(base) == len(oof) and np.allclose(base[[f"y_{l}" for l in LAB]].values, oof[[f"y_{l}" for l in LAB]].values, atol=1e-6), "fixed labels differ"
fold = oof.fold.values
def frank(df):                                    # rank within each fold (models are compared on rank, as at inference)
    out = {}
    for l in LAB:
        p = df[f"p_{l}"].values.astype(float); r = np.zeros(len(p))
        for k in np.unique(fold): m = fold == k; r[m] = rankdata(p[m]) / m.sum()
        out[l] = r
    return out
def auc_l(y, p):
    m = (y >= 0.9) | (y <= 0.1)
    return roc_auc_score((y[m] >= 0.9).astype(int), p[m]) if m.sum() > 20 and len({*(y[m] >= 0.9)}) == 2 else np.nan
Y = {l: base[f"y_{l}"].values for l in LAB}
arms = {"baseline": base, TAG: oof}
CX = list(Path("/kaggle/input").rglob("oof_v6convnext.csv"))
assert CX, "attach the FIRST ConvNeXt run's output (oof_v6convnext.csv) for the seed-ensemble check"
cx = pd.read_csv(CX[0]); assert len(cx) == len(base); arms["convnext(seed42)"] = cx; print("first-run OOF:", CX[0])
R = {n: frank(d) for n, d in arms.items()}
R["ens 2 seeds"] = {l: (R["convnext(seed42)"][l] + R[TAG][l]) / 2 for l in LAB}
T = pd.DataFrame({n: {l: auc_l(Y[l], r[l]) for l in LAB} for n, r in R.items()})
T.loc["MACRO"] = T.loc[LAB].mean(); T.loc["KEY-4"] = T.loc[KEY4].mean()
print("All 5 folds pooled with per-fold ranks:\n", T.round(3).to_string())
print("\nper-fold macro (raw predictions):")
for k in range(5):
    m = fold == k
    f = lambda df: np.nanmean([auc_l(Y[l][m], df.loc[m, f"p_{l}"].values) for l in LAB])
    print(f"  fold {k}: baseline {f(base):.4f} | {TAG} {f(oof):.4f} | delta {f(oof)-f(base):+.4f}")
g = oof.is_gold.values.astype(bool)
if g.sum() > 10:
    gm = lambda r: np.nanmean([roc_auc_score((Y[l][g] >= 0.5).astype(int), r[l][g]) for l in LAB if len({*(Y[l][g] >= 0.5)}) == 2])
    print(f"\ngold ({int(g.sum())} studies, CIs about +/-0.1, NOT decisive): " + " | ".join(f"{n} {gm(r):.3f}" for n, r in R.items()))
rng = np.random.default_rng(0); d = []
for _ in range(300):
    b = rng.integers(0, len(base), len(base))
    d.append(np.nanmean([auc_l(Y[l][b], R[TAG][l][b]) - auc_l(Y[l][b], R["baseline"][l][b]) for l in KEY4]))
print(f"\nKEY-4 {TAG} - baseline: {T.loc['KEY-4', TAG]-T.loc['KEY-4','baseline']:+.4f}  90% CI [{np.percentile(d,5):+.4f}, {np.percentile(d,95):+.4f}] (study resampling only; seed noise on one fold was ~0.005 KEY-4, ~0.01 for the v6 config)")
print("\nseed check (pooled, per-fold ranks):")
for n in ("convnext(seed42)", TAG, "ens 2 seeds"): print(f"  {n:20s} MACRO {T.loc['MACRO', n]:.4f}  KEY-4 {T.loc['KEY-4', n]:.4f}")
print("the two single seeds should agree within ~0.01; the 2-seed ensemble is what sub9 submits")

first-run OOF: /kaggle/input/notebooks/theokorir/knee-v6-convnext-5fold/oof_v6convnext.csv
All 5 folds pooled with per-fold ranks:
                   baseline  v6convnext2  convnext(seed42)  ens 2 seeds
ACL                  0.781        0.858             0.853        0.860
MCL                  0.725        0.795             0.797        0.801
Medial Meniscus      0.770        0.885             0.881        0.887
Lateral Meniscus     0.736        0.796             0.786        0.795
Medial OA            0.826        0.856             0.860        0.860
Lateral OA           0.747        0.769             0.777        0.777
PF OA                0.717        0.720             0.727        0.727
Effusion             0.822        0.844             0.840        0.845
Synovitis            0.772        0.792             0.791        0.793
Baker's              0.756        0.855             0.850        0.857
Contusion            0.755        0.830             0.824        0.831
Fracture        